## Google Colab Setup

Run the cell below once when using Colab. It installs dependencies and checks for GPU.

In [17]:
import subprocess, sys

# Install dependencies (safe to run on Colab or locally)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "torch", "sentencepiece", "pandas", "numpy", "tqdm"],
    check=False,
)

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("Running on CPU — consider switching to a GPU runtime on Colab.")

GPU available: False
Running on CPU — consider switching to a GPU runtime on Colab.


# Assignment 4: Prompt Engineering for NER

**Student:** MingHsiang Lee  
**Task:** Named Entity Recognition (NER) with local CoNLL-2003

## 0-shot vs Few-shot

- **0-shot**: instruction + input only, no examples.
- **Few-shot**: instruction + a few labeled examples + input.

This notebook compares both approaches.

In [18]:
import json
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Fast mode is for quick iteration while tuning prompts/parsing.
USE_FAST_MODE = True
FAST_SENTENCE_LIMIT = 300  # Good first check; usually around 1 hour on local CPU.
FULL_SENTENCE_LIMIT = None  # Set an integer if you want to cap full mode.

# Local CPU-safe models.
MODEL_NAMES = [
    "google/flan-t5-small",  # 80M params
    "google/flan-t5-base",   # 250M params
]

# Beam search width and generation length.
NUM_BEAMS = 4
MAX_NEW_TOKENS = 128

# Keep every run artifact (no overwrite).
OUTPUT_DIR = Path("assignment4_runs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# Relative paths only (no absolute path).
CONLL_REL_PATHS = [
    Path("Assignment2_Name_Entity_Recognition/conll2003"),
    Path("../Assignment2_Name_Entity_Recognition/conll2003"),
]

CONLL_DIR = None
for p in CONLL_REL_PATHS:
    if p.exists():
        CONLL_DIR = p
        break

if CONLL_DIR is None:
    raise FileNotFoundError("Could not find local CoNLL folder via relative paths.")

EVAL_FILE = "eng.testa"
print("Run ID:", RUN_ID)
print("Output directory:", OUTPUT_DIR.resolve())
print("Using CoNLL directory:", CONLL_DIR)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
print("Mode:", "FAST" if USE_FAST_MODE else "FULL")

Run ID: 20260317_134106
Output directory: /Users/Thomas/Desktop/Nat_Lang_Engr_Meth_Tools/Assignment4_Prompt Engineering/assignment4_runs
Using CoNLL directory: ../Assignment2_Name_Entity_Recognition/conll2003
Device: CPU
Mode: FAST


In [19]:
def read_conll(file_path: Path):
    rows = []
    tokens, tags = [], []
    with file_path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                if tokens:
                    rows.append({"tokens": tokens, "tags": tags})
                    tokens, tags = [], []
                continue
            if line.startswith("-DOCSTART-"):
                continue
            parts = line.split()
            if len(parts) >= 4:
                tokens.append(parts[0])
                tags.append(parts[-1])
    if tokens:
        rows.append({"tokens": tokens, "tags": tags})
    return rows


def bio_to_entities(tokens, tags):
    out = []
    i = 0
    while i < len(tags):
        tag = tags[i]
        if tag == "O" or "-" not in tag:
            i += 1
            continue
        prefix, label = tag.split("-", 1)
        if prefix not in {"B", "I"}:
            i += 1
            continue
        ent = [tokens[i]]
        i += 1
        while i < len(tags) and tags[i] == f"I-{label}":
            ent.append(tokens[i])
            i += 1
        out.append({"entity": " ".join(ent), "type": label})
    return out


sentences = read_conll(CONLL_DIR / EVAL_FILE)
data = []
for s in sentences:
    text = " ".join(s["tokens"])
    gold_entities = bio_to_entities(s["tokens"], s["tags"])
    data.append({"text": text, "gold_entities": gold_entities})

eval_df = pd.DataFrame(data)

if USE_FAST_MODE:
    eval_df = eval_df.head(FAST_SENTENCE_LIMIT).copy()
elif FULL_SENTENCE_LIMIT is not None:
    eval_df = eval_df.head(FULL_SENTENCE_LIMIT).copy()

print("Mode:", "FAST" if USE_FAST_MODE else "FULL")
print("Sentence count:", len(eval_df))
display(eval_df.head(3))

Mode: FAST
Sentence count: 300


,text,gold_entities
0,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]"
1,LONDON 1996-08-30,"[{'entity': 'LONDON', 'type': 'LOC'}]"
2,West Indian all-rounder Phil Simmons took four...,"[{'entity': 'West Indian', 'type': 'MISC'}, {'..."


In [20]:
PROMPTS = {
    "zero_shot_plain": (
        "Named Entity Recognition task.\n"
        "Extract all named entities from the text and label each as PER, ORG, LOC, or MISC.\n"
        "Return ONLY a valid JSON array of objects with keys entity and type.\n"
        "The text may be in ALL CAPS. Copy exact entity spans from the text.\n"
        "Valid example: [{\"entity\":\"LONDON\",\"type\":\"LOC\"}]\n"
        "Invalid example: PER\n"
        "Invalid example: [\"entity\":\"LONDON\",\"type\":\"LOC\"]\n"
        "If no entities, return [].\n"
        "Text: {text}\n"
        "JSON:"
    ),
    "zero_shot_rubric": (
        "You are a strict NER extractor.\n"
        "PER=person, ORG=organization/team/company, LOC=location/place, MISC=other named entity.\n"
        "Output must be valid JSON array only. No explanation, no code, no labels alone.\n"
        "Use this exact structure for each item: {\"entity\":\"NAME\",\"type\":\"PER|ORG|LOC|MISC\"}.\n"
        "Valid example output: [{\"entity\":\"LONDON\",\"type\":\"LOC\"}]\n"
        "If uncertain, return [].\n"
        "Text: {text}\n"
        "JSON:"
    ),
    "few_shot_caps": (
        "Extract named entities and return only a valid JSON array.\n"
        "\n"
        "Text: LEICESTERSHIRE BEAT NOTTINGHAMSHIRE IN CRICKET FINAL .\n"
        "Output: [{\"entity\":\"LEICESTERSHIRE\",\"type\":\"ORG\"},{\"entity\":\"NOTTINGHAMSHIRE\",\"type\":\"ORG\"}]\n"
        "\n"
        "Text: LONDON 1996-08-30\n"
        "Output: [{\"entity\":\"LONDON\",\"type\":\"LOC\"}]\n"
        "\n"
        "Text: UNITED STATES BEAT GERMANY 2-1 .\n"
        "Output: [{\"entity\":\"UNITED STATES\",\"type\":\"LOC\"},{\"entity\":\"GERMANY\",\"type\":\"LOC\"}]\n"
        "\n"
        "Text: {text}\n"
        "Output:"
    ),
    "few_shot_balanced": (
        "Extract entities from text. Types: PER, ORG, LOC, MISC.\n"
        "Return valid JSON array only. Do not output PER/ORG/LOC/MISC alone.\n"
        "\n"
        "Text: THE MATCH WAS PLAYED IN PARIS .\n"
        "Output: [{\"entity\":\"PARIS\",\"type\":\"LOC\"}]\n"
        "\n"
        "Text: JOHN SMITH JOINS REUTERS IN NEW YORK .\n"
        "Output: [{\"entity\":\"JOHN SMITH\",\"type\":\"PER\"},{\"entity\":\"REUTERS\",\"type\":\"ORG\"},{\"entity\":\"NEW YORK\",\"type\":\"LOC\"}]\n"
        "\n"
        "Text: THE WEATHER WAS FINE YESTERDAY .\n"
        "Output: []\n"
        "\n"
        "Text: {text}\n"
        "Output:"
    ),
}


def build_prompt(name, text):
    return PROMPTS[name].replace("{text}", text.strip())

In [21]:
def build_generator(model_name):
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return {"tokenizer": tokenizer, "model": model, "device": device}


generators = {}
for model_name in MODEL_NAMES:
    print(f"Loading {model_name} ...")
    generators[model_name] = build_generator(model_name)

print("Loaded models:", list(generators.keys()))

Loading google/flan-t5-small ...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1243.30it/s, Materializing param=shared.weight]                                                      
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading google/flan-t5-base ...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1110.54it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded models: ['google/flan-t5-small', 'google/flan-t5-base']


In [22]:
import re

ALLOWED_TYPES = {"PER", "ORG", "LOC", "MISC"}


def _normalize_loaded_json(loaded):
    if isinstance(loaded, dict):
        loaded = [loaded]
    if not isinstance(loaded, list):
        return []

    entities = []
    for item in loaded:
        if not isinstance(item, dict):
            continue
        ent = str(item.get("entity", "")).strip()
        typ = str(item.get("type", "")).strip().upper()
        if ent and ent != "..." and typ in ALLOWED_TYPES:
            entities.append({"entity": ent, "type": typ})
    return entities


def _recover_entities_fallback(text):
    if not text:
        return []

    recovered = []
    seen = set()

    # entity before type
    pattern_1 = re.compile(
        r'"entity"\s*:\s*"([^"]+)"\s*,\s*"type"\s*:\s*"?(PER|ORG|LOC|MISC)"?',
        flags=re.IGNORECASE,
    )
    for match in pattern_1.finditer(text):
        ent = match.group(1).strip()
        typ = match.group(2).upper().strip()
        key = (" ".join(ent.lower().split()), typ)
        if ent and ent != "..." and typ in ALLOWED_TYPES and key not in seen:
            recovered.append({"entity": ent, "type": typ})
            seen.add(key)

    # type before entity
    pattern_2 = re.compile(
        r'"type"\s*:\s*"?(PER|ORG|LOC|MISC)"?\s*,\s*"entity"\s*:\s*"([^"]+)"',
        flags=re.IGNORECASE,
    )
    for match in pattern_2.finditer(text):
        typ = match.group(1).upper().strip()
        ent = match.group(2).strip()
        key = (" ".join(ent.lower().split()), typ)
        if ent and ent != "..." and typ in ALLOWED_TYPES and key not in seen:
            recovered.append({"entity": ent, "type": typ})
            seen.add(key)

    return recovered


def parse_ner_output(raw_text):
    text = (raw_text or "").strip()
    left = text.find("[")
    right = text.rfind("]")
    candidate = text[left : right + 1] if left != -1 and right != -1 and right > left else text

    strict_json_ok = False
    entities = []

    try:
        loaded = json.loads(candidate)
        entities = _normalize_loaded_json(loaded)
        strict_json_ok = True
    except Exception:
        strict_json_ok = False

    # Recover entities from common malformed outputs without counting as valid JSON.
    if not entities:
        entities = _recover_entities_fallback(candidate)

    parse_source = "strict" if strict_json_ok else ("fallback" if entities else "none")
    return entities, strict_json_ok, parse_source


def to_entity_set(items):
    out = set()
    for item in items:
        ent = " ".join(str(item.get("entity", "")).lower().split())
        typ = str(item.get("type", "")).upper().strip()
        if ent and typ in ALLOWED_TYPES:
            out.add((ent, typ))
    return out


def generate_text(gen, prompt, max_new_tokens=MAX_NEW_TOKENS):
    tokenizer = gen["tokenizer"]
    model = gen["model"]
    model_device = gen["device"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def evaluate_prompt_model(df, prompt_name, model_name):
    gen = generators[model_name]

    tp = fp = fn = 0
    valid = exact = 0
    rows = []

    for row_idx, row in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f"{model_name} | {prompt_name}")):
        prompt = build_prompt(prompt_name, row.text)
        raw = generate_text(gen, prompt)

        pred_entities, parse_ok, parse_source = parse_ner_output(raw)
        valid += int(parse_ok)

        gold_set = to_entity_set(row.gold_entities)
        pred_set = to_entity_set(pred_entities)

        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)
        exact += int(gold_set == pred_set)

        rows.append(
            {
                "example_id": row_idx,
                "text": row.text,
                "gold_entities": row.gold_entities,
                "pred_entities": pred_entities,
                "parse_ok": parse_ok,
                "parse_source": parse_source,
                "raw_output": raw,
            }
        )

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    metrics = {
        "model": model_name,
        "prompt_template": prompt_name,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "valid_json_rate": round(valid / len(df), 4),
        "exact_sentence_match": round(exact / len(df), 4),
    }

    return metrics, pd.DataFrame(rows)

In [23]:
all_metrics = []
all_predictions = {}

for model_name in MODEL_NAMES:
    for prompt_name in PROMPTS.keys():
        metrics_row, pred_df = evaluate_prompt_model(eval_df, prompt_name, model_name)
        all_metrics.append(metrics_row)
        all_predictions[(model_name, prompt_name)] = pred_df

metrics_df = (
    pd.DataFrame(all_metrics)
    .sort_values(by=["f1", "precision", "recall"], ascending=False)
    .reset_index(drop=True)
)

display(metrics_df)

metrics_file = OUTPUT_DIR / f"metrics_{RUN_ID}.csv"
metrics_latest = OUTPUT_DIR / "metrics_latest.csv"
metrics_df.to_csv(metrics_file, index=False)
metrics_df.to_csv(metrics_latest, index=False)

print("Saved metrics:", metrics_file)
print("Updated latest metrics:", metrics_latest)

google/flan-t5-small | zero_shot_plain: 100%|██████████| 300/300 [00:47<00:00,  6.38it/s]
google/flan-t5-small | zero_shot_rubric: 100%|██████████| 300/300 [01:30<00:00,  3.30it/s]
google/flan-t5-small | few_shot_caps: 100%|██████████| 300/300 [02:33<00:00,  1.96it/s]
google/flan-t5-small | few_shot_balanced: 100%|██████████| 300/300 [01:38<00:00,  3.04it/s]
google/flan-t5-base | zero_shot_plain: 100%|██████████| 300/300 [07:32<00:00,  1.51s/it]
google/flan-t5-base | zero_shot_rubric: 100%|██████████| 300/300 [11:01<00:00,  2.21s/it]
google/flan-t5-base | few_shot_caps: 100%|██████████| 300/300 [10:10<00:00,  2.04s/it]
google/flan-t5-base | few_shot_balanced: 100%|██████████| 300/300 [09:53<00:00,  1.98s/it]


,model,prompt_template,precision,recall,f1,valid_json_rate,exact_sentence_match
0,google/flan-t5-base,few_shot_caps,0.8000,0.0075,0.0150,0.0233,0.1900
1,google/flan-t5-small,few_shot_caps,0.5714,0.0075,0.0149,0.4167,0.1900
2,google/flan-t5-base,few_shot_balanced,0.5000,0.0057,0.0112,0.0433,0.1800
3,google/flan-t5-small,zero_shot_plain,0.0000,0.0000,0.0000,0.6633,0.1767
4,google/flan-t5-small,zero_shot_rubric,0.0000,0.0000,0.0000,0.7967,0.1767
5,google/flan-t5-small,few_shot_balanced,0.0000,0.0000,0.0000,0.3633,0.1767
6,google/flan-t5-base,zero_shot_plain,0.0000,0.0000,0.0000,0.3067,0.1767
7,google/flan-t5-base,zero_shot_rubric,0.0000,0.0000,0.0000,0.1167,0.1633


Saved metrics: assignment4_runs/metrics_20260317_134106.csv
Updated latest metrics: assignment4_runs/metrics_latest.csv


In [24]:
shot_df = metrics_df.copy()
shot_df["shot_type"] = shot_df["prompt_template"].apply(
    lambda x: "few-shot" if x.startswith("few_shot") else "zero-shot"
)
shot_df = (
    shot_df.groupby(["model", "shot_type"], as_index=False)[
        ["precision", "recall", "f1", "valid_json_rate", "exact_sentence_match"]
    ]
    .mean()
    .sort_values(by=["model", "shot_type"])
)

print("0-shot vs few-shot summary:")
display(shot_df)

shot_file = OUTPUT_DIR / f"shot_summary_{RUN_ID}.csv"
shot_latest = OUTPUT_DIR / "shot_summary_latest.csv"
shot_df.to_csv(shot_file, index=False)
shot_df.to_csv(shot_latest, index=False)

# Build full logs from all evaluated predictions (no re-run and no truncation).
log_frames = []
for (model_name, prompt_name), pred_df in all_predictions.items():
    temp = pred_df.copy()
    temp.insert(1, "model", model_name)
    temp.insert(2, "prompt_template", prompt_name)
    log_frames.append(temp)

logs_df = pd.concat(log_frames, ignore_index=True)

column_order = [
    "example_id",
    "model",
    "prompt_template",
    "parse_ok",
    "parse_source",
    "text",
    "gold_entities",
    "pred_entities",
    "raw_output",
]
logs_df = logs_df[column_order]

display(logs_df.head(20))

logs_file = OUTPUT_DIR / f"ner_prompt_output_logs_{RUN_ID}.csv"
logs_latest = OUTPUT_DIR / "ner_prompt_output_logs_latest.csv"
logs_df.to_csv(logs_file, index=False)
logs_df.to_csv(logs_latest, index=False)

parse_summary_df = (
    logs_df.groupby(["model", "prompt_template", "parse_source"], as_index=False)
    .size()
    .sort_values(by=["model", "prompt_template", "parse_source"])
)
parse_summary_file = OUTPUT_DIR / f"parse_source_summary_{RUN_ID}.csv"
parse_summary_latest = OUTPUT_DIR / "parse_source_summary_latest.csv"
parse_summary_df.to_csv(parse_summary_file, index=False)
parse_summary_df.to_csv(parse_summary_latest, index=False)

print("Saved shot summary:", shot_file)
print("Saved full logs:", logs_file)
print("Saved parse-source summary:", parse_summary_file)
print("Updated latest files in:", OUTPUT_DIR.resolve())

0-shot vs few-shot summary:


,model,shot_type,precision,recall,f1,valid_json_rate,exact_sentence_match
0,google/flan-t5-base,few-shot,0.6500,0.00660,0.01310,0.0333,0.18500
1,google/flan-t5-base,zero-shot,0.0000,0.00000,0.00000,0.2117,0.17000
2,google/flan-t5-small,few-shot,0.2857,0.00375,0.00745,0.3900,0.18335
3,google/flan-t5-small,zero-shot,0.0000,0.00000,0.00000,0.7300,0.17670


,example_id,model,prompt_template,parse_ok,parse_source,text,gold_entities,pred_entities,raw_output
0,0,google/flan-t5-small,zero_shot_plain,True,strict,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],[]
1,1,google/flan-t5-small,zero_shot_plain,True,strict,LONDON 1996-08-30,"[{'entity': 'LONDON', 'type': 'LOC'}]",[],[]
2,2,google/flan-t5-small,zero_shot_plain,False,none,West Indian all-rounder Phil Simmons took four...,"[{'entity': 'West Indian', 'type': 'MISC'}, {'...",[],"PER, ORG, LOC"
3,3,google/flan-t5-small,zero_shot_plain,True,strict,"Their stay on top , though , may be short-live...","[{'entity': 'Essex', 'type': 'ORG'}, {'entity'...",[],[]
4,4,google/flan-t5-small,zero_shot_plain,True,strict,After bowling Somerset out for 83 on the openi...,"[{'entity': 'Somerset', 'type': 'ORG'}, {'enti...",[],[]
5,5,google/flan-t5-small,zero_shot_plain,True,strict,"Trailing by 213 , Somerset got a solid start t...","[{'entity': 'Somerset', 'type': 'ORG'}, {'enti...",[],[]
6,6,google/flan-t5-small,zero_shot_plain,True,strict,"Essex , however , look certain to regain their...","[{'entity': 'Essex', 'type': 'ORG'}, {'entity'...",[],[]
7,7,google/flan-t5-small,zero_shot_plain,True,strict,"Hussain , considered surplus to England 's one...","[{'entity': 'Hussain', 'type': 'PER'}, {'entit...",[],[]
8,8,google/flan-t5-small,zero_shot_plain,True,strict,By the close Yorkshire had turned that into a ...,"[{'entity': 'Yorkshire', 'type': 'ORG'}, {'ent...",[],[]
9,9,google/flan-t5-small,zero_shot_plain,True,strict,"At the Oval , Surrey captain Chris Lewis , ano...","[{'entity': 'Oval', 'type': 'LOC'}, {'entity':...",[],[]


Saved shot summary: assignment4_runs/shot_summary_20260317_134106.csv
Saved full logs: assignment4_runs/ner_prompt_output_logs_20260317_134106.csv
Saved parse-source summary: assignment4_runs/parse_source_summary_20260317_134106.csv
Updated latest files in: /Users/Thomas/Desktop/Nat_Lang_Engr_Meth_Tools/Assignment4_Prompt Engineering/assignment4_runs


## Bias and Mitigation

Potential bias sources in prompt-based NER:
- Entity overfitting to frequent names/regions in news.
- Prompt-example imbalance by entity type.
- Domain shift from news text to other domains.

Mitigation:
1. Use balanced few-shot examples.
2. Include no-entity examples (`[]`) to reduce false positives.
3. Evaluate per entity type and per domain.
4. Enforce strict output schema and parse checks.

### Deliverable Coverage
- Multiple prompt designs: yes
- 0-shot vs few-shot: yes
- Multi-model comparison: yes
- Output logs: yes
- Bias discussion: yes